# Quick check — Task 3: simulation-based EMD prediction

Runs the causal (denoised) prediction pipeline (`src/emd_prediction/`) on a synthetic population: predict the sequence distribution 270 days after a prediction date, and score each method against the observed future distribution by EMD (earth mover's distance, Hamming ground metric) and consensus Hamming distance.

Methods: GenModel / GenModel-Average (MPL selection + Wright–Fisher medoid), MLR, and the baselines Null, Lower Bound, Null-Emp (raw observations) and Null-Emp-Denoised. SLM is switched off here.

Requirements: the HiGHS LP solver (for the EMD). Small settings are used (`K = 10`, `M = 5`, `N = 500`, against `K = 1000`, `M = 100`, `N = 1000` in production). Runtime: about 1 minute, plus compilation on the first run. Start Julia with several threads to speed up the medoid step.

In [ ]:
# Run from the notebooks/ folder (or the project root)
const PROJECT_ROOT = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
using Pkg; Pkg.activate(PROJECT_ROOT)
using Dates, DataFrames, Statistics, Random

include(joinpath(PROJECT_ROOT, "src", "emd_prediction", "mutation_simulation.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "FluStudy.jl"))
include(joinpath(PROJECT_ROOT, "src", "emd_prediction", "PredictionPipeline.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "synthetic_data.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "cli.jl"))                    # run_name (output file names)
using .FluStudy, .PredictionPipeline, .SyntheticFlu

## 1. Synthetic input

`simulate_population` (`src/shared/synthetic_data.jl`) evolves a population with the negative-binomial dynamics of `src/shared/population_simulation.jl`, using its default parameters:

* 20 sites with 4 states each; selection coefficients `s ~ Normal(0, 0.05)`, with 0 for the alleles of a random reference sequence;
* 100 random starting sequences sharing 1000 individuals;
* offspring numbers negative binomial with R = 2 and dispersion k = 0.1, population ceiling 10 000;
* mutation rate 10⁻³ per site per generation;
* 1000 generations, one generation per day (2009-01-02 … 2011-09-28).

Each day 20 individuals are sampled and "sequenced" (states 1–4 are written as the amino acids A, C, D, E). The samples are saved as a FASTA file with the same header layout as the real data, then read back with `FluStudy.create_study`, as the command-line scripts do (`protein_length = 20` instead of 566 for HA).

In [ ]:
syn = simulate_population(seed = 1, n_sample = 20)
fasta = joinpath(mktempdir(), "synthetic_20sites.fasta")
write_fasta(fasta, syn)

start_date, stop_date = Date(2009, 1, 1), Date(2009, 1, 1) + Day(1000)
study = redirect_stdout(devnull) do
    FluStudy.create_study(fasta; protein = "HA", protein_length = syn.L,
                          start_date = start_date, stop_date = stop_date, tstep = 10.0, tsmooth = 60.0)
end
data = study.data
println("population size: median ", round(Int, median(syn.pop_size)), ", range ", extrema(syn.pop_size),
        " | distinct sequences in the population: median ", round(Int, median(syn.n_species)))
println("selection coefficients: ", count(>(0.05), syn.s), " alleles with s > 0.05, ", count(<(-0.05), syn.s), " with s < -0.05 (of ", length(syn.s), ")")

In [ ]:
# FluStudy reproduces the synthetic samples: the same (day, count, sequence) rows, up to their order
# within a day. It drops the last FASTA date, because its daily bins are right-open.
sorted_rows(m) = sort(Tuple.(eachrow(m)))
@assert sorted_rows(data) == sorted_rows(syn.data[syn.data[:, 1] .< maximum(syn.data[:, 1]), :])
println("data matrix: ", size(data), "  (rows × [day, count, $(syn.L) sites])")

## 2. Predict 270 days ahead of 2010-06-01

The same function as `run_emd_prediction.jl`. Every step uses only information up to its own anchor date. Day numbers follow the convention of the real data: day *d* is 2009-01-01 + *d* days.

In [ ]:
Random.seed!(5)
obs_date, Δ = Date(2010, 6, 1), 270
res = redirect_stdout(devnull) do
    run_prediction_pipeline_short_ext_denoised(data, obs_date, Δ, "H3N2";
        slm = false, K = 10, M = 5, N = 500, freq_threshold = 1e-4, medoid_threshold = 1e-4,
        user_start_date = start_date, verbose = false)
end
println("prediction date ", res.date1, " (day ", res.t1, ")  ->  target date ", res.date2, " (day ", res.t2, ")")
scores = distance_summary_table_short_ext(res)

In [ ]:
# Sanity checks
emd_of(m) = only(scores[scores.Method .== m, "EMD(Pop)"])
@assert Set(scores.Method) == Set(["Null", "GenModel", "GenModel-Average", "Lower Bound", "Null-Emp", "MLR", "Null-Emp-Denoised"])
@assert all(isfinite, scores[!, "EMD(Pop)"]) && all(>=(0), scores[!, "EMD(Pop)"])
@assert emd_of("Lower Bound") <= emd_of("Null") + 1e-9        # the lower bound is the best transport onto t1 sequences
@assert res.date2 - res.date1 <= Day(Δ + 30)
@assert sum(values(res.d1)) ≈ 1 && sum(values(res.d2)) ≈ 1
println("distinct sequences: d1 = ", length(res.d1), ", d2 = ", length(res.d2), ", GenModel output = ", length(res.output))

## 3. Save the run

`save_pipeline_run` writes the summary (one row per method) and the side-car files: the predicted distribution of each method, d1/d2/dlb, the empirical bins, and the selection coefficients. File names end in the same tag as `run_emd_prediction.jl` uses (`<subtype>_<protein>_<date>_h<horizon>`, plus `_init<date>` when the fit does not start at day 0); the production script writes them to `outputs/emd_prediction/`.

In [ ]:
out_dir = mktempdir()
tag = run_name("", "H3N2", "HA", obs_date; h = Δ)
paths = save_pipeline_run(res, out_dir; subtype = "H3N2", protein = "HA", region = "synthetic",
                          obs_date = obs_date, horizon = Δ, tag = tag)
@assert isfile(paths.summary)
println(join(sort(readdir(out_dir)), "\n"))